# EXP-009 LLM labels + Axial fold0

唯一变量: 标签 → steven **llm_labels_v4_blend**（gold 优先）
对照: Axial+labels005=**0.699**；S+C+A+labels005=**0.721**
- rsna_v3 `dc2a25d4994bfd62280af5ca06bfce64eb8cafd7e89a58dcd6a5de735c905a5a`（含 model.pt + LLM 填充）
- 目标: 验证 LLM 标签是否最大杠杆；预算 ~2h GPU


In [ ]:
# 提交模式:禁网(enable_internet:false),torch 2.2.0+cu118 + nvidia-cu11 依赖全部离线安装
# 坑1:Kaggle 挂载把文件名中 '+' 去掉(2.2.0cu118=非法 PEP440)→ 复制为合法名
# 坑2:torch 2.2.0+cu118 的 CUDA 库在 nvidia-*-cu11 依赖包中(不在 torch wheel 内),
#     只装 torch 会缺 libcudart(v8/v9 实证)→ 必须显式安装全部 13 个 wheel
# 坑3:/kaggle/working 会被收集为 kernel 输出 → 装完立即删除
import subprocess, sys, os, glob, shutil
wh = sorted(glob.glob('/kaggle/input/**/*.whl', recursive=True))
print('挂载 wheel 共', len(wh), '个')
for w in wh: print('  ', os.path.basename(w))
assert len(wh) >= 13, 'FAIL: 离线 wheel 不足(应 13: torch+torchvision+11 nvidia-cu11)'
fixed = []
for w in wh:
    base = os.path.basename(w)
    dst = os.path.join('/kaggle/working', base.replace('cu118', '+cu118', 1) if 'cu118' in base else base)
    shutil.copy(w, dst)
    fixed.append(dst)
print('安装 wheel:', [os.path.basename(f) for f in fixed])
subprocess.check_call([sys.executable,"-m","pip","uninstall","-y","-q","torch","torchvision","torchaudio"])
# 显式安装全部 wheel(--no-deps --no-index: 无版本解析/无联网)
subprocess.check_call([sys.executable,"-m","pip","install","-q","--no-deps","--no-index"] + fixed)
for f in fixed: os.remove(f)
import numpy, torch, torchvision, sklearn, pandas
print("torch", torch.__version__, "cuda", torch.version.cuda, "| numpy", numpy.__version__)
assert torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# preflight: Axial only
import os, csv
base='/kaggle/input'
def _find(b, depth=0, max_depth=3):
    if not os.path.isdir(b): return None
    try: entries=sorted(os.listdir(b))
    except OSError: return None
    for e in entries:
        p=os.path.join(b,e)
        if not os.path.isdir(p): continue
        if e in ('train_series','test_series'): continue
        if os.path.isfile(os.path.join(p,'test.csv')) and os.path.isdir(os.path.join(p,'test_series')):
            return p
        if depth<max_depth-1:
            r=_find(p,depth+1)
            if r: return r
    return None
root=_find(base)
print('root', root)
assert root
# LLM labels present?
hits=[]
for dp,dn,fns in os.walk('/kaggle/input'):
    for fn in fns:
        if 'llm_labels' in fn.lower() and fn.endswith('.csv'):
            hits.append(os.path.join(dp,fn))
print('llm hits', hits)
assert any('v4_blend' in h or 'llm_labels' in h for h in hits), 'no LLM labels dataset mounted'
print('preflight OK')


In [ ]:
import base64, os, hashlib
parts=[]; parts.append("...")
path="/kaggle/working/rsna_v3.py"
open(path,"w",encoding="utf-8").write(base64.b64decode(parts[0]).decode("utf-8"))
h=hashlib.sha256(open(path,"rb").read()).hexdigest()
print("rsna_v3",h); assert h=="dc2a25d4994bfd62280af5ca06bfce64eb8cafd7e89a58dcd6a5de735c905a5a"
assert "_load_llm_labels" in open(path,encoding="utf-8").read()
print("exp009 v3 OK")


In [ ]:
import subprocess, sys, os, json
os.chdir("/kaggle/working")
cmd=[sys.executable,"-u","rsna_v3.py","--planes","axial","--epochs","3","--folds","5","--only-fold","0",
     "--batch","8","--n-clips","2","--experiment-id","exp_009_llm_axial_fold0","--fuse","mean","--verbose"]
print("=== EXP-009 LLM labels + Axial fold0 ===", flush=True)
print(" ".join(cmd), flush=True)
r=subprocess.run(cmd)
print("returncode", r.returncode, flush=True)
root="artifacts/exp_009_llm_axial_fold0/fold0/Axial"
assert os.path.isfile(os.path.join(root,"model.pt")), "no model.pt"
assert os.path.isfile(os.path.join(root,"test_pred.npy"))
print("weights+pred ready")
